# ECG-Centric PCOS Screening System

## Pre-Model Validation and Biomarker Verification Pipeline

This notebook removes the leakage-prone `Diagnosis` column, revalidates selected biomarkers, verifies ECG feature importance and separability, compares ECG-only versus physiological-only performance, and produces publication-quality outputs.

Input expected in Google Colab: `best_bayesian_150.csv`

In [ ]:
# STEP 0: Install required packages
!pip -q install -U scikit-learn boruta umap-learn

In [ ]:
# STEP 0: Imports, configuration, and output folders
import os
import json
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from scipy.stats import mannwhitneyu
from sklearn.base import clone
from sklearn.compose import ColumnTransformer
from sklearn.decomposition import PCA
from sklearn.ensemble import RandomForestClassifier
from sklearn.feature_selection import mutual_info_classif
from sklearn.impute import SimpleImputer
from sklearn.inspection import permutation_importance
from sklearn.manifold import TSNE
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    silhouette_score,
)
from sklearn.model_selection import StratifiedKFold
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import LabelEncoder, OrdinalEncoder, StandardScaler

from boruta import BorutaPy
import umap

warnings.filterwarnings("ignore")

RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)

INPUT_FILE = Path("best_bayesian_150.csv")
OUTPUT_DIR = Path("outputs")
REPORT_DIR = OUTPUT_DIR / "reports"
PLOT_DIR = OUTPUT_DIR / "plots"
TABLE_DIR = OUTPUT_DIR / "tables"
for directory in [OUTPUT_DIR, REPORT_DIR, PLOT_DIR, TABLE_DIR]:
    directory.mkdir(parents=True, exist_ok=True)

sns.set_theme(
    context="paper",
    style="whitegrid",
    palette="Set2",
    rc={
        "figure.dpi": 120,
        "savefig.dpi": 300,
        "axes.spines.top": False,
        "axes.spines.right": False,
        "axes.titleweight": "bold",
        "font.size": 11,
    },
)

def savefig(path, tight=True):
    if tight:
        plt.tight_layout()
    plt.savefig(path, dpi=300, bbox_inches="tight")
    plt.close()

def normalize_name(name):
    return (
        str(name)
        .strip()
        .lower()
        .replace(" ", "_")
        .replace("-", "_")
        .replace("/", "_")
        .replace("(", "")
        .replace(")", "")
    )

def find_column(df, candidates):
    normalized = {normalize_name(c): c for c in df.columns}
    for candidate in candidates:
        key = normalize_name(candidate)
        if key in normalized:
            return normalized[key]
    return None

def likely_identifier_columns(df, target_col):
    ids = []
    for col in df.columns:
        key = normalize_name(col)
        if col == target_col:
            continue
        if key in {"id", "subject_id", "sample_id", "record_id", "patient_id"} or key.endswith("_id"):
            ids.append(col)
    return ids

def prepare_xy(df, feature_cols, target_col):
    X_raw = df[feature_cols].copy()
    y_raw = df[target_col].copy()

    target_encoder = LabelEncoder()
    y = target_encoder.fit_transform(y_raw.astype(str).fillna("__missing__"))

    numeric_cols = [c for c in feature_cols if pd.api.types.is_numeric_dtype(X_raw[c])]
    categorical_cols = [c for c in feature_cols if c not in numeric_cols]

    numeric_pipe = Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
    ])
    categorical_pipe = Pipeline([
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("encoder", OrdinalEncoder(handle_unknown="use_encoded_value", unknown_value=-1)),
    ])

    transformers = []
    if numeric_cols:
        transformers.append(("num", numeric_pipe, numeric_cols))
    if categorical_cols:
        transformers.append(("cat", categorical_pipe, categorical_cols))

    preprocessor = ColumnTransformer(transformers=transformers, remainder="drop", verbose_feature_names_out=False)
    X = pd.DataFrame(
        preprocessor.fit_transform(X_raw),
        columns=preprocessor.get_feature_names_out(),
        index=df.index,
    ).astype(float)

    return X, y, target_encoder, preprocessor, numeric_cols, categorical_cols

def get_positive_label_index(label_encoder):
    classes = [str(c) for c in label_encoder.classes_]
    for idx, label in enumerate(classes):
        if "pcos" in label.lower() or label.strip() == "1":
            return idx
    return 1 if len(classes) > 1 else 0

def safe_roc_auc(y_true, y_score):
    try:
        return roc_auc_score(y_true, y_score)
    except Exception:
        return np.nan

print("Environment ready.")
print(f"Outputs will be saved under: {OUTPUT_DIR.resolve()}")

## STEP 1 - Data Sanity Check

In [ ]:
# STEP 1: Load data, remove Diagnosis leakage, and generate sanity report
if not INPUT_FILE.exists():
    raise FileNotFoundError(
        f"{INPUT_FILE} was not found. Upload best_bayesian_150.csv into the Colab working directory and rerun."
    )

df_original = pd.read_csv(INPUT_FILE)
df = df_original.copy()

print("Original dataset shape:", df.shape)

diagnosis_cols = [c for c in df.columns if normalize_name(c) == "diagnosis"]
if diagnosis_cols:
    print("Removing leakage-prone column(s):", diagnosis_cols)
    df = df.drop(columns=diagnosis_cols)
else:
    print("No Diagnosis column found. Leakage guard passed.")

remaining_diagnosis_cols = [c for c in df.columns if normalize_name(c) == "diagnosis"]
assert not remaining_diagnosis_cols, "Diagnosis leakage column is still present."

target_col = find_column(df, ["Target_Class"])
if target_col is None:
    raise ValueError("Target_Class column is required and was not found.")

class_distribution = df[target_col].value_counts(dropna=False).rename_axis("Class").reset_index(name="Count")
class_distribution["Percent"] = 100 * class_distribution["Count"] / len(df)

sanity_report = pd.DataFrame({
    "Column": df.columns,
    "DType": [str(df[c].dtype) for c in df.columns],
    "Missing_Count": [df[c].isna().sum() for c in df.columns],
    "Missing_Percent": [100 * df[c].isna().mean() for c in df.columns],
    "Unique_Count": [df[c].nunique(dropna=True) for c in df.columns],
})
sanity_report.to_csv(REPORT_DIR / "data_sanity_report.csv", index=False)
class_distribution.to_csv(REPORT_DIR / "class_distribution.csv", index=False)

print("\nDataset shape after leakage removal:", df.shape)
print("\nMissing values:")
display(df.isna().sum().to_frame("Missing_Count"))
print("\nClass distribution:")
display(class_distribution)
print("\nData types:")
display(df.dtypes.to_frame("DType"))
print("\nSaved:", REPORT_DIR / "data_sanity_report.csv")

## STEP 2 - Re-Run Feature Selection Without Diagnosis

In [ ]:
# STEP 2: Define clean feature matrix
identifier_cols = likely_identifier_columns(df, target_col)
feature_cols = [c for c in df.columns if c not in [target_col] + identifier_cols]

if any(normalize_name(c) == "diagnosis" for c in feature_cols):
    raise RuntimeError("Diagnosis found in feature columns. Stop: leakage risk.")

X, y, target_encoder, preprocessor, numeric_cols, categorical_cols = prepare_xy(df, feature_cols, target_col)
positive_idx = get_positive_label_index(target_encoder)

print("Target column:", target_col)
print("Target classes:", list(target_encoder.classes_))
print("Identifier columns excluded:", identifier_cols)
print("Feature count:", len(feature_cols))
print("Numeric features:", len(numeric_cols), "| Categorical features:", len(categorical_cols))

In [ ]:
# STEP 2A: Correlation filter at |r| > 0.90
CORR_THRESHOLD = 0.90

corr_matrix = X.corr(method="pearson")
corr_matrix.to_csv(TABLE_DIR / "correlation_matrix.csv")

mean_abs_corr = corr_matrix.abs().replace(1.0, np.nan).mean(skipna=True).fillna(0)
flagged_pairs = []
features_to_remove = set()

for i, left in enumerate(corr_matrix.columns):
    for right in corr_matrix.columns[i + 1:]:
        r = corr_matrix.loc[left, right]
        if abs(r) > CORR_THRESHOLD:
            remove = left if mean_abs_corr[left] >= mean_abs_corr[right] else right
            keep = right if remove == left else left
            features_to_remove.add(remove)
            flagged_pairs.append({
                "Feature_1": left,
                "Feature_2": right,
                "Pearson_r": r,
                "Abs_r": abs(r),
                "Recommended_Remove": remove,
                "Recommended_Keep": keep,
            })

correlation_filtered_features = pd.DataFrame(flagged_pairs)
if correlation_filtered_features.empty:
    correlation_filtered_features = pd.DataFrame(
        columns=["Feature_1", "Feature_2", "Pearson_r", "Abs_r", "Recommended_Remove", "Recommended_Keep"]
    )
correlation_filtered_features.to_csv(TABLE_DIR / "correlation_filtered_features.csv", index=False)

retained_features = [c for c in X.columns if c not in features_to_remove]
pd.DataFrame({"Retained_Feature": retained_features}).to_csv(TABLE_DIR / "retained_features_after_correlation.csv", index=False)

plt.figure(figsize=(12, 10))
sns.heatmap(corr_matrix, cmap="vlag", center=0, square=True, linewidths=0.25, cbar_kws={"label": "Pearson r"})
plt.title("Feature Correlation Heatmap")
plt.xlabel("Features")
plt.ylabel("Features")
savefig(PLOT_DIR / "correlation_heatmap.png")

print(f"Correlation matrix saved. Features flagged for removal: {len(features_to_remove)}")
display(correlation_filtered_features)

In [ ]:
# STEP 2B: Mutual information feature ranking
mi_scores = mutual_info_classif(X, y, discrete_features=False, random_state=RANDOM_SEED)
mi_df = (
    pd.DataFrame({"Feature": X.columns, "Mutual_Information": mi_scores})
    .sort_values("Mutual_Information", ascending=False)
    .reset_index(drop=True)
)
mi_df["MI_Rank"] = np.arange(1, len(mi_df) + 1)
mi_df.to_csv(TABLE_DIR / "mutual_information_scores.csv", index=False)

plt.figure(figsize=(10, max(5, 0.35 * len(mi_df))))
sns.barplot(data=mi_df, y="Feature", x="Mutual_Information", color="#3B82F6")
plt.title("Mutual Information Feature Ranking")
plt.xlabel("Mutual Information Score")
plt.ylabel("Feature")
savefig(PLOT_DIR / "mutual_information_ranking.png")

display(mi_df)

In [ ]:
# STEP 2C: Boruta feature selection
rf_for_boruta = RandomForestClassifier(
    n_estimators=1000,
    class_weight="balanced",
    random_state=RANDOM_SEED,
    n_jobs=-1,
    max_depth=None,
)

boruta_selector = BorutaPy(
    estimator=rf_for_boruta,
    n_estimators="auto",
    max_iter=100,
    perc=100,
    alpha=0.05,
    two_step=True,
    random_state=RANDOM_SEED,
    verbose=0,
)

boruta_selector.fit(X.values, y)

boruta_status = np.where(
    boruta_selector.support_,
    "Confirmed",
    np.where(boruta_selector.support_weak_, "Tentative", "Rejected"),
)
boruta_df = pd.DataFrame({
    "Feature": X.columns,
    "Boruta Status": boruta_status,
    "Boruta Ranking": boruta_selector.ranking_,
}).sort_values(["Boruta Ranking", "Feature"]).reset_index(drop=True)
boruta_df.to_csv(TABLE_DIR / "boruta_results.csv", index=False)

status_order = ["Confirmed", "Tentative", "Rejected"]
plot_boruta = boruta_df.copy()
plot_boruta["Status_Score"] = plot_boruta["Boruta Status"].map({"Confirmed": 3, "Tentative": 2, "Rejected": 1})
plot_boruta = plot_boruta.sort_values(["Status_Score", "Boruta Ranking"], ascending=[False, True])

plt.figure(figsize=(10, max(5, 0.35 * len(plot_boruta))))
sns.barplot(
    data=plot_boruta,
    y="Feature",
    x="Status_Score",
    hue="Boruta Status",
    hue_order=status_order,
    dodge=False,
    palette={"Confirmed": "#16A34A", "Tentative": "#F59E0B", "Rejected": "#9CA3AF"},
)
plt.xticks([1, 2, 3], ["Rejected", "Tentative", "Confirmed"])
plt.xlabel("Boruta Status")
plt.ylabel("Feature")
plt.title("Boruta Feature Selection Status")
plt.legend(title="Status", loc="lower right")
savefig(PLOT_DIR / "boruta_status_plot.png")

print("Boruta confirmed features:", boruta_df.loc[boruta_df["Boruta Status"] == "Confirmed", "Feature"].tolist())
print("Boruta tentative features:", boruta_df.loc[boruta_df["Boruta Status"] == "Tentative", "Feature"].tolist())
display(boruta_df)

## STEP 3 - Random Forest Feature Validation

In [ ]:
# STEP 3: Temporary Random Forest validation with stratified 5-fold CV
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_SEED)
rf_validation = RandomForestClassifier(
    n_estimators=600,
    random_state=RANDOM_SEED,
    class_weight="balanced",
    n_jobs=-1,
)

rf_importances = []
perm_importances = []

for fold, (train_idx, test_idx) in enumerate(cv.split(X, y), start=1):
    X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
    y_train, y_test = y[train_idx], y[test_idx]
    model = clone(rf_validation)
    model.fit(X_train, y_train)
    rf_importances.append(model.feature_importances_)

    perm = permutation_importance(
        model,
        X_test,
        y_test,
        scoring="f1_weighted",
        n_repeats=30,
        random_state=RANDOM_SEED + fold,
        n_jobs=-1,
    )
    perm_importances.append(perm.importances_mean)

rf_mean = np.mean(rf_importances, axis=0)
perm_mean = np.mean(perm_importances, axis=0)

feature_validation = pd.DataFrame({
    "Feature": X.columns,
    "Boruta Status": boruta_df.set_index("Feature").loc[X.columns, "Boruta Status"].values,
    "Boruta Ranking": boruta_df.set_index("Feature").loc[X.columns, "Boruta Ranking"].values,
    "RF Importance": rf_mean,
    "Permutation Importance": perm_mean,
})
feature_validation = feature_validation.sort_values(
    ["Boruta Ranking", "RF Importance", "Permutation Importance"],
    ascending=[True, False, False],
).reset_index(drop=True)
feature_validation.to_csv(TABLE_DIR / "feature_validation_table.csv", index=False)

rf_plot = feature_validation.sort_values("RF Importance", ascending=False)
plt.figure(figsize=(10, max(5, 0.35 * len(rf_plot))))
sns.barplot(data=rf_plot, y="Feature", x="RF Importance", color="#2563EB")
plt.title("Random Forest Feature Importance")
plt.xlabel("Mean Gini Importance Across 5 Folds")
plt.ylabel("Feature")
savefig(PLOT_DIR / "rf_feature_importance.png")

perm_plot = feature_validation.sort_values("Permutation Importance", ascending=False)
plt.figure(figsize=(10, max(5, 0.35 * len(perm_plot))))
sns.barplot(data=perm_plot, y="Feature", x="Permutation Importance", color="#DC2626")
plt.axvline(0, color="black", linewidth=0.8)
plt.title("Permutation Importance")
plt.xlabel("Mean Decrease in Weighted F1 Across 5 Folds")
plt.ylabel("Feature")
savefig(PLOT_DIR / "permutation_importance.png")

display(feature_validation)

## STEP 4 - Biomarker Consistency Analysis

In [ ]:
# STEP 4: Evaluate leading ECG biomarkers
biomarker_candidates = {
    "QRS Duration": ["QRS Duration", "QRS_Duration", "QRS_Duration (s)"],
    "SDNN_MeanRR_Ratio": ["SDNN_MeanRR_Ratio"],
    "T_Wave_Duration": ["T_Wave_Duration", "T Wave Duration", "T_Wave_Duration (s)"],
    "QTc_Bazett": ["QTc_Bazett", "QTc Bazett", "QTc_Bazett (s)"],
}

biomarker_cols = {}
for display_name, candidates in biomarker_candidates.items():
    col = find_column(df, candidates)
    if col is not None and col in feature_cols:
        biomarker_cols[display_name] = col

if not biomarker_cols:
    raise ValueError("None of the requested ECG biomarkers were found in the dataset.")

label_series = pd.Series(target_encoder.inverse_transform(y), index=df.index, name="Class")
biomarker_records = []

for display_name, col in biomarker_cols.items():
    values = pd.to_numeric(df[col], errors="coerce")
    groups = []
    group_names = []
    for class_name in target_encoder.classes_:
        group_values = values[label_series == str(class_name)].dropna()
        groups.append(group_values)
        group_names.append(str(class_name))

    if len(groups) == 2 and all(len(g) > 0 for g in groups):
        stat, p_value = mannwhitneyu(groups[0], groups[1], alternative="two-sided")
        n1, n2 = len(groups[0]), len(groups[1])
        rank_biserial = 1 - (2 * stat) / (n1 * n2)
    else:
        stat, p_value, rank_biserial = np.nan, np.nan, np.nan

    biomarker_records.append({
        "Biomarker": display_name,
        "Column": col,
        "Class_1": group_names[0] if group_names else "",
        "Class_2": group_names[1] if len(group_names) > 1 else "",
        "Mann_Whitney_U": stat,
        "p_value": p_value,
        "Rank_Biserial_Effect_Size": rank_biserial,
        "Median_Class_1": groups[0].median() if groups else np.nan,
        "Median_Class_2": groups[1].median() if len(groups) > 1 else np.nan,
    })

biomarker_report = pd.DataFrame(biomarker_records).sort_values("p_value")
biomarker_report.to_csv(REPORT_DIR / "biomarker_consistency_report.csv", index=False)

plot_df = df[[target_col] + list(biomarker_cols.values())].copy()
plot_df[target_col] = label_series.values
long_df = plot_df.melt(id_vars=target_col, var_name="Biomarker_Column", value_name="Value")
name_lookup = {v: k for k, v in biomarker_cols.items()}
long_df["Biomarker"] = long_df["Biomarker_Column"].map(name_lookup)
long_df["Value"] = pd.to_numeric(long_df["Value"], errors="coerce")

g = sns.displot(
    data=long_df.dropna(),
    x="Value",
    hue=target_col,
    col="Biomarker",
    kind="kde",
    fill=True,
    common_norm=False,
    col_wrap=2,
    height=3.2,
    aspect=1.25,
)
g.fig.suptitle("Biomarker Distributions by Class", y=1.03, fontweight="bold")
g.set_axis_labels("Value", "Density")
g.savefig(PLOT_DIR / "biomarker_distribution_plots.png", dpi=300, bbox_inches="tight")
plt.close(g.fig)

plt.figure(figsize=(11, 6))
sns.violinplot(data=long_df.dropna(), x="Biomarker", y="Value", hue=target_col, split=False, inner="quartile")
plt.title("ECG Biomarker Violin Plots")
plt.xlabel("Biomarker")
plt.ylabel("Value")
plt.xticks(rotation=20, ha="right")
savefig(PLOT_DIR / "biomarker_violin_plots.png")

plt.figure(figsize=(11, 6))
sns.boxplot(data=long_df.dropna(), x="Biomarker", y="Value", hue=target_col, showfliers=False)
sns.stripplot(data=long_df.dropna(), x="Biomarker", y="Value", hue=target_col, dodge=True, size=2, alpha=0.35, legend=False)
plt.title("ECG Biomarker Boxplots")
plt.xlabel("Biomarker")
plt.ylabel("Value")
plt.xticks(rotation=20, ha="right")
savefig(PLOT_DIR / "biomarker_boxplots.png")

display(biomarker_report)

## STEP 5 - Class Separation Visualization

In [ ]:
# STEP 5: PCA, UMAP, and t-SNE using retained ECG features
physiological_candidates = {
    "BMI": ["BMI"],
    "Cycle Length": ["Cycle Length", "Cycle_Length"],
    "Cycle Regularity": ["Cycle Regularity", "Cycle_Regularity"],
    "Family History": ["Family History", "Family History (PCOS/Diabetes)", "Family_History"],
}

physiological_cols = []
for candidates in physiological_candidates.values():
    col = find_column(pd.DataFrame(columns=X.columns), candidates)
    if col is not None:
        physiological_cols.append(col)
physiological_cols = list(dict.fromkeys(physiological_cols))

retained_features = [c for c in X.columns if c not in features_to_remove]
ecg_retained_features = [c for c in retained_features if c not in physiological_cols]

if len(ecg_retained_features) < 2:
    raise ValueError("At least two retained ECG features are required for 2D separation visualizations.")

X_ecg_scaled = StandardScaler().fit_transform(X[ecg_retained_features])
labels_text = target_encoder.inverse_transform(y)

def plot_projection(coords, title, path, subtitle=None):
    proj_df = pd.DataFrame({"Dim 1": coords[:, 0], "Dim 2": coords[:, 1], "Class": labels_text})
    plt.figure(figsize=(7, 6))
    sns.scatterplot(
        data=proj_df,
        x="Dim 1",
        y="Dim 2",
        hue="Class",
        style="Class",
        s=65,
        edgecolor="white",
        linewidth=0.5,
        alpha=0.9,
    )
    full_title = title if subtitle is None else f"{title}\n{subtitle}"
    plt.title(full_title)
    plt.xlabel("Component 1")
    plt.ylabel("Component 2")
    plt.legend(title="Class", frameon=True)
    savefig(path)

pca = PCA(n_components=2, random_state=RANDOM_SEED)
pca_coords = pca.fit_transform(X_ecg_scaled)
pca_var = pca.explained_variance_ratio_
pca_silhouette = silhouette_score(pca_coords, y) if len(np.unique(y)) > 1 else np.nan
plot_projection(
    pca_coords,
    "PCA Projection of Retained ECG Features",
    PLOT_DIR / "pca_visualization.png",
    f"Explained variance: PC1={pca_var[0]:.2%}, PC2={pca_var[1]:.2%}; silhouette={pca_silhouette:.3f}",
)

umap_model = umap.UMAP(n_components=2, random_state=RANDOM_SEED, n_neighbors=min(15, max(2, len(df) - 1)), min_dist=0.1)
umap_coords = umap_model.fit_transform(X_ecg_scaled)
umap_silhouette = silhouette_score(umap_coords, y) if len(np.unique(y)) > 1 else np.nan
plot_projection(
    umap_coords,
    "UMAP Projection of Retained ECG Features",
    PLOT_DIR / "umap_visualization.png",
    f"silhouette={umap_silhouette:.3f}",
)

perplexity = min(30, max(5, (len(df) - 1) // 3))
tsne = TSNE(n_components=2, random_state=RANDOM_SEED, init="pca", learning_rate="auto", perplexity=perplexity)
tsne_coords = tsne.fit_transform(X_ecg_scaled)
tsne_silhouette = silhouette_score(tsne_coords, y) if len(np.unique(y)) > 1 else np.nan
plot_projection(
    tsne_coords,
    "t-SNE Projection of Retained ECG Features",
    PLOT_DIR / "tsne_visualization.png",
    f"perplexity={perplexity}; silhouette={tsne_silhouette:.3f}",
)

separation_report = pd.DataFrame([
    {"Method": "PCA", "Silhouette": pca_silhouette, "Explained_Variance_PC1": pca_var[0], "Explained_Variance_PC2": pca_var[1]},
    {"Method": "UMAP", "Silhouette": umap_silhouette, "Explained_Variance_PC1": np.nan, "Explained_Variance_PC2": np.nan},
    {"Method": "t-SNE", "Silhouette": tsne_silhouette, "Explained_Variance_PC1": np.nan, "Explained_Variance_PC2": np.nan},
])
separation_report.to_csv(REPORT_DIR / "class_separation_report.csv", index=False)
display(separation_report)

## STEP 6 - ECG vs Physiological Comparison

In [ ]:
# STEP 6: ECG-only vs physiological-only vs combined retained features
missing_phys = [name for name, candidates in physiological_candidates.items() if find_column(pd.DataFrame(columns=X.columns), candidates) is None]
print("Physiological columns used:", physiological_cols)
if missing_phys:
    print("Warning: missing physiological feature(s):", missing_phys)

combined_features = retained_features
feature_groups = {
    "Physiological Only": physiological_cols,
    "ECG Only": ecg_retained_features,
    "Combined Retained": combined_features,
}

def evaluate_rf_cv(X_data, y_data, group_name):
    if X_data.shape[1] == 0:
        return {
            "Feature Group": group_name,
            "Feature Count": 0,
            "Accuracy": np.nan,
            "Precision": np.nan,
            "Recall": np.nan,
            "F1": np.nan,
            "ROC-AUC": np.nan,
        }

    metrics = []
    for fold, (train_idx, test_idx) in enumerate(cv.split(X_data, y_data), start=1):
        model = RandomForestClassifier(
            n_estimators=600,
            random_state=RANDOM_SEED + fold,
            class_weight="balanced",
            n_jobs=-1,
        )
        X_train, X_test = X_data.iloc[train_idx], X_data.iloc[test_idx]
        y_train, y_test = y_data[train_idx], y_data[test_idx]
        model.fit(X_train, y_train)
        pred = model.predict(X_test)
        if hasattr(model, "predict_proba") and len(model.classes_) == 2:
            proba = model.predict_proba(X_test)[:, list(model.classes_).index(positive_idx)]
        else:
            proba = pred

        metrics.append({
            "Accuracy": accuracy_score(y_test, pred),
            "Precision": precision_score(y_test, pred, average="binary" if len(np.unique(y_data)) == 2 else "weighted", pos_label=positive_idx, zero_division=0),
            "Recall": recall_score(y_test, pred, average="binary" if len(np.unique(y_data)) == 2 else "weighted", pos_label=positive_idx, zero_division=0),
            "F1": f1_score(y_test, pred, average="binary" if len(np.unique(y_data)) == 2 else "weighted", pos_label=positive_idx, zero_division=0),
            "ROC-AUC": safe_roc_auc(y_test, proba) if len(np.unique(y_data)) == 2 else np.nan,
        })

    metric_df = pd.DataFrame(metrics)
    result = {"Feature Group": group_name, "Feature Count": X_data.shape[1]}
    for col in ["Accuracy", "Precision", "Recall", "F1", "ROC-AUC"]:
        result[col] = metric_df[col].mean()
        result[f"{col}_Std"] = metric_df[col].std(ddof=1)
    return result

comparison_rows = []
for group_name, cols in feature_groups.items():
    comparison_rows.append(evaluate_rf_cv(X[cols], y, group_name))

comparison_df = pd.DataFrame(comparison_rows)
comparison_df.to_csv(TABLE_DIR / "feature_group_comparison.csv", index=False)

plot_metrics = comparison_df.melt(
    id_vars=["Feature Group", "Feature Count"],
    value_vars=["Accuracy", "Precision", "Recall", "F1", "ROC-AUC"],
    var_name="Metric",
    value_name="Score",
)
plt.figure(figsize=(10, 6))
sns.barplot(data=plot_metrics, x="Metric", y="Score", hue="Feature Group")
plt.ylim(0, 1.05)
plt.title("ECG vs Physiological Feature Group Performance")
plt.xlabel("Metric")
plt.ylabel("5-Fold CV Score")
plt.legend(title="Feature Group", loc="lower right")
savefig(PLOT_DIR / "performance_comparison_plot.png")

display(comparison_df)

## STEP 7 - Project Objective Assessment

In [ ]:
# STEP 7: Objective assessment
confirmed_features = boruta_df.loc[boruta_df["Boruta Status"] == "Confirmed", "Feature"].tolist()
tentative_features = boruta_df.loc[boruta_df["Boruta Status"] == "Tentative", "Feature"].tolist()

top_ecg_features = [
    f for f in feature_validation.sort_values("Permutation Importance", ascending=False)["Feature"].tolist()
    if f in ecg_retained_features
]
significant_biomarkers = biomarker_report.loc[biomarker_report["p_value"] < 0.05, "Biomarker"].tolist()

ecg_perf = comparison_df.loc[comparison_df["Feature Group"] == "ECG Only", "ROC-AUC"].iloc[0]
phys_perf = comparison_df.loc[comparison_df["Feature Group"] == "Physiological Only", "ROC-AUC"].iloc[0]
combined_perf = comparison_df.loc[comparison_df["Feature Group"] == "Combined Retained", "ROC-AUC"].iloc[0]

objective_1 = "ACHIEVED" if len(ecg_retained_features) > 0 and "Diagnosis" not in df.columns else "NOT ACHIEVED"
objective_2 = "ACHIEVED" if len(significant_biomarkers) >= 2 or len(top_ecg_features[:4]) >= 4 else "NOT ACHIEVED"
objective_3 = "ACHIEVED" if len(confirmed_features) + len(tentative_features) > 0 else "NOT ACHIEVED"

ready_conditions = [
    objective_1 == "ACHIEVED",
    objective_2 == "ACHIEVED",
    objective_3 == "ACHIEVED",
    not np.isnan(ecg_perf),
    ecg_perf >= 0.70,
    combined_perf >= max(ecg_perf, phys_perf) - 0.05 if not np.isnan(phys_perf) else True,
]
objective_4 = "READY" if all(ready_conditions) else "NOT READY"
definitive_recommendation = "READY FOR MODEL TRAINING" if objective_4 == "READY" else "ADDITIONAL FEATURE VALIDATION REQUIRED"

objective_report = pd.DataFrame([
    {"Objective": "Create ECG-centric PCOS dataset", "Status": objective_1, "Evidence": f"{len(ecg_retained_features)} retained ECG features; Diagnosis removed."},
    {"Objective": "Identify ECG biomarkers", "Status": objective_2, "Evidence": f"Significant biomarkers: {', '.join(significant_biomarkers) if significant_biomarkers else 'None at p<0.05'}."},
    {"Objective": "Obtain final feature subset", "Status": objective_3, "Evidence": f"{len(confirmed_features)} confirmed and {len(tentative_features)} tentative Boruta features."},
    {"Objective": "Ready for model training", "Status": objective_4, "Evidence": f"ECG-only ROC-AUC={ecg_perf:.3f}; combined ROC-AUC={combined_perf:.3f}."},
])
objective_report.to_csv(REPORT_DIR / "objective_assessment_report.csv", index=False)
display(objective_report)
print(definitive_recommendation)

## STEP 8 - Publication-Quality Figure Inventory

In [ ]:
# STEP 8: Verify required publication-quality figures exist
required_figures = [
    "correlation_heatmap.png",
    "mutual_information_ranking.png",
    "boruta_status_plot.png",
    "rf_feature_importance.png",
    "permutation_importance.png",
    "pca_visualization.png",
    "umap_visualization.png",
    "tsne_visualization.png",
    "biomarker_boxplots.png",
    "biomarker_violin_plots.png",
    "performance_comparison_plot.png",
]

figure_inventory = pd.DataFrame({
    "Figure": required_figures,
    "Path": [str(PLOT_DIR / f) for f in required_figures],
    "Exists": [(PLOT_DIR / f).exists() for f in required_figures],
})
figure_inventory.to_csv(REPORT_DIR / "figure_inventory.csv", index=False)
display(figure_inventory)

if not figure_inventory["Exists"].all():
    missing = figure_inventory.loc[~figure_inventory["Exists"], "Figure"].tolist()
    raise RuntimeError(f"Missing required figure(s): {missing}")

## STEP 9 - Final Executive Summary

In [ ]:
# STEP 9: Generate executive summary
top_ecg_biomarkers = top_ecg_features[:6]
phys_cols_present = [c for c in physiological_cols if c in feature_validation["Feature"].values]
top_physiological = (
    feature_validation[feature_validation["Feature"].isin(phys_cols_present)]
    .sort_values("Permutation Importance", ascending=False)["Feature"]
    .tolist()
)

def metric_line(group):
    row = comparison_df.loc[comparison_df["Feature Group"] == group].iloc[0]
    return (
        f"Accuracy={row['Accuracy']:.3f}, Precision={row['Precision']:.3f}, "
        f"Recall={row['Recall']:.3f}, F1={row['F1']:.3f}, ROC-AUC={row['ROC-AUC']:.3f}"
    )

pca_quality = "strong" if pca_silhouette >= 0.50 else "moderate" if pca_silhouette >= 0.25 else "limited"
umap_quality = "strong" if umap_silhouette >= 0.50 else "moderate" if umap_silhouette >= 0.25 else "limited"

summary = f"""ECG-Centric PCOS Screening System: Pre-Model Validation Executive Summary

Input dataset: {INPUT_FILE}
Rows: {df.shape[0]}
Columns after Diagnosis leakage removal: {df.shape[1]}
Target label: {target_col}
Class labels: {list(target_encoder.classes_)}

1. Top ECG Biomarkers
{', '.join(top_ecg_biomarkers) if top_ecg_biomarkers else 'No ECG biomarkers ranked above validation threshold.'}

2. Top Physiological Biomarkers
{', '.join(top_physiological) if top_physiological else 'No physiological biomarkers available or retained.'}

3. PCA Separation Quality
PCA silhouette={pca_silhouette:.3f}; explained variance PC1={pca_var[0]:.2%}, PC2={pca_var[1]:.2%}. Separation quality: {pca_quality}.

4. UMAP Separation Quality
UMAP silhouette={umap_silhouette:.3f}. Separation quality: {umap_quality}.

5. ECG-only Performance
{metric_line('ECG Only')}

6. Physiology-only Performance
{metric_line('Physiological Only')}

7. Combined Performance
{metric_line('Combined Retained')}

8. Recommendation
{definitive_recommendation}

Rationale:
- Diagnosis was removed before all feature selection, visualization, and evaluation steps.
- Target_Class was used only as the target label.
- ECG-only performance was directly compared against physiological-only and combined retained feature sets.
- Boruta, RF importance, and permutation importance were consolidated in feature_validation_table.csv.
"""

(REPORT_DIR / "executive_summary.txt").write_text(summary)
print(summary)

## STEP 10 - Export Manifest

In [ ]:
# STEP 10: Export manifest
all_outputs = []
for folder in [REPORT_DIR, TABLE_DIR, PLOT_DIR]:
    for path in sorted(folder.glob("*")):
        all_outputs.append({
            "File": path.name,
            "Path": str(path),
            "Category": folder.name,
            "Size_Bytes": path.stat().st_size,
        })

manifest = pd.DataFrame(all_outputs)
manifest.to_csv(OUTPUT_DIR / "export_manifest.csv", index=False)
display(manifest)

print("\nPipeline complete.")
print("Definitive recommendation:", definitive_recommendation)
print("All outputs saved under:", OUTPUT_DIR.resolve())